# AV2 주행 사건 시간 관계 빈도 분석 (RefAV val, 정답 트랙)

목적: 사건 종류와 평가셋을 확정하기 전에, AV2 val 로그에서 **같은 차량의 두 사건이 BEFORE / MEETS / OVERLAP 관계로 나타나는 빈도**를 셉니다.

- 입력: RefAV 정답 주석(`sm_annotations.feather` 또는 `scenario_mining_val_annotations.feather`) + AV2 ego pose(`city_SE3_egovehicle.feather`)
- 카메라·LiDAR는 쓰지 않습니다. 마지막 선택 셀만 GT 검수용으로 카메라 이미지 몇 장을 내려받습니다.
- 출력(`OUT_DIR`): `events.csv`, `instances.csv`, `summary_*.csv`, `kinematics.feather`, `params.json`

사건 판별 기준은 RefAV 원자 함수의 기준값을 따랐습니다.

| 사건 | 판별 | RefAV 근거 |
|---|---|---|
| STOP | 속력 < 0.5 m/s 가 1.0 s 이상 | `has_velocity` 설명: 정지 객체도 지터로 0.5 m/s까지 나옴 |
| DECELERATE | 종방향 가속도 ≤ −1.0 m/s² 가 0.5 s 이상 | `accelerating` 설명: −1 미만이면 제동 |
| ACCELERATE | 종방향 가속도 ≥ 1.0 m/s² 가 0.5 s 이상 | `accelerating` 설명: 1.0 초과면 가속 |
| TURN | 요레이트 ≥ 5°/s 인 같은 방향 구간의 누적 회전 ≥ 45° | `turning()`의 `ANG_VEL_THRESH`, `TURN_ANGLE_THRESH` |
| (제외) 주차 차량 | 전체 관측 중 이동 범위 < 3 m | `stationary()` |

관계 정의: 두 구간 중 먼저 시작한 것을 first, 간격 gap = second.start − first.end
- **OVERLAP**: 교집합 > 0.5 s
- **MEETS**: −0.5 s ≤ gap ≤ 0.2 s (맞닿음. 감속이 끝나며 정지하는 경우 등)
- **BEFORE**: gap > 0.2 s, 요약표에서는 gap ≤ Δ(기본 3 s)만 셈

**이 기준값은 개발 단계 설정입니다.** 빈도를 보고 조정하되, 평가셋을 만든 뒤에는 고정해야 합니다.

In [11]:
# ==== 0. 설치 및 기본 설정 ====
!pip -q install boto3 pyarrow scipy tqdm huggingface_hub

import os, sys, json, time
from pathlib import Path
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm
from IPython.display import display
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

SPLIT = 'val'
OUT_DIR = Path('/content/drive/MyDrive/refav_temporal/freq' if IN_COLAB else './freq_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# RefAV get_objects_of_category(..., 'VEHICLE')의 상위 범주 목록과 동일
VEHICLE_CATS = ['EGO_VEHICLE', 'ARTICULATED_BUS', 'BOX_TRUCK', 'BUS', 'LARGE_VEHICLE', 'CAR',
                'MOTORCYCLE', 'RAILED_VEHICLE', 'REGULAR_VEHICLE', 'SCHOOL_BUS', 'TRUCK', 'TRUCK_CAB']
INCLUDE_EGO = True   # ego 차량도 하나의 트랙으로 포함 (RefAV 질의에 ego가 자주 등장)

P = dict(
    V_STOP=0.5,          # m/s
    STOP_MIN_DUR=1.0,    # s
    A_DECEL=-1.0,        # m/s^2
    A_ACCEL=1.0,         # m/s^2
    ACC_MIN_DUR=0.5,     # s
    MIN_DV=1.5,          # m/s  감속/가속 구간 동안의 속력 변화 하한
    TURN_YAWRATE=5.0,    # deg/s
    TURN_ANGLE=45.0,     # deg
    TURN_MIN_DIST=5.0,   # m  회전 구간 동안 이동 거리 하한 (정지 중 요 지터 배제)
    TURN_GAP_MERGE=2.0,  # s  같은 방향 회전 구간 사이의 끊김 허용 (회전 중 잠깐 정지 포함)
    GAP_FILL=0.3,        # s  STOP/DECEL/ACCEL 구간의 짧은 끊김 메움
    PARKED_DISP=3.0,     # m
    SEG_GAP=0.25,        # s  이보다 긴 관측 공백이면 트랙을 분할
    MIN_SEG_FRAMES=11,   # 약 1.1 s 미만 조각은 미분하지 않음
    SMOOTH_WIN=1.1,      # s  Savitzky-Golay 창
    MEET_TOL=0.2,        # s  간격이 이 이하이면 BEFORE가 아니라 MEETS(맞닿음). 평활화로 경계가 0.1~0.2 s 흔들림
    MIN_OVERLAP=0.5,     # s  교집합이 이보다 커야 OVERLAP. 이하의 짧은 겹침은 MEETS로 봄
    DELTAS=[1.0, 3.0, 5.0],
    DELTA_MAIN=3.0,
    MIN_LOGS_FEASIBLE=5, # 관계별 서로 다른 로그 수가 이 이상이면 대조쌍 구성 가능으로 표시
)
EVENT_TYPES = ['TURN', 'STOP', 'DECELERATE', 'ACCELERATE']

def s3_client():
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config
    return boto3.client('s3', region_name='us-east-1',
                        config=Config(signature_version=UNSIGNED, max_pool_connections=32))
print('OUT_DIR =', OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUT_DIR = /content/drive/MyDrive/refav_temporal/freq


In [12]:
# ==== 1. 정답 주석 위치 찾기 ====
# 경로를 알면 직접 지정하십시오. None이면 자동 탐색합니다.
SM_LOG_ROOT = None   # 예: '/content/drive/MyDrive/RefAV/output/sm_dataset/val'  (<log_id>/sm_annotations.feather)
SM_FEATHER  = None   # 예: '/content/drive/MyDrive/RefAV/scenario_mining_downloads/scenario_mining_val_annotations.feather'
SEARCH_ROOTS = ['/content/drive/MyDrive', '/content'] if IN_COLAB else ['.']
SKIP_DIRS = {'sensors', 'cache', 'crops', '.git', '__pycache__', 'sm_predictions',
             'tracker_predictions', 'tracker_downloads', 'llm_code_predictions', 'ego_poses'}
MIN_LOGS = 100 if SPLIT == 'val' else 1

def walk_find(roots, filename, max_depth=6):
    hits = []
    for root in map(Path, roots):
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).parts) - base >= max_depth:
                dirnames[:] = []
            dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
            if filename in filenames:
                hits.append(Path(dirpath) / filename)
    return hits

def download_sm_feather():
    name = f'scenario_mining_{SPLIT}_annotations.feather'
    dst_dir = OUT_DIR.parent / 'scenario_mining_downloads'
    dst_dir.mkdir(parents=True, exist_ok=True)
    try:
        from huggingface_hub import list_repo_files, hf_hub_download
        files = [f for f in list_repo_files('CainanD/RefAV', repo_type='dataset') if f.endswith(name)]
        if files:
            print('Hugging Face에서 다운로드:', files[0])
            return Path(hf_hub_download('CainanD/RefAV', files[0], repo_type='dataset', local_dir=str(dst_dir)))
    except Exception as e:
        print('Hugging Face 실패:', repr(e)[:200])
    s3 = s3_client()
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket='argoverse', Prefix='tasks/scenario_mining/'):
        for o in page.get('Contents', []):
            if o['Key'].endswith(name):
                dst = dst_dir / name
                print('Argoverse S3에서 다운로드:', o['Key'])
                s3.download_file('argoverse', o['Key'], str(dst))
                return dst
    raise FileNotFoundError(name)

if SM_LOG_ROOT is None and SM_FEATHER is None:
    t_start = time.time()
    per_log = walk_find(SEARCH_ROOTS, 'sm_annotations.feather')
    roots = pd.Series([str(p.parent.parent) for p in per_log], dtype=str).value_counts()
    print('sm_annotations.feather 상위 폴더 후보 (로그 수):')
    print(roots.to_string() if len(roots) else '  없음')
    ok = [r for r in roots.index if roots[r] >= MIN_LOGS and 'sm_dataset' in r]
    if ok:
        SM_LOG_ROOT = Path(ok[0])
    else:
        big = walk_find(SEARCH_ROOTS, f'scenario_mining_{SPLIT}_annotations.feather')
        SM_FEATHER = big[0] if big else download_sm_feather()
    print(f'탐색 {time.time() - t_start:.0f}s')

print('SM_LOG_ROOT =', SM_LOG_ROOT)
print('SM_FEATHER  =', SM_FEATHER)

sm_annotations.feather 상위 폴더 후보 (로그 수):
  없음
탐색 1s
SM_LOG_ROOT = None
SM_FEATHER  = /content/drive/MyDrive/refav_temporal/scenario_mining_downloads/scenario_mining_val_annotations.feather


In [13]:
# ==== 2. 차량 트랙 주석 로드 ====
import pyarrow as pa
import pyarrow.ipc as ipc

COLS = ['track_uuid', 'category', 'timestamp_ns', 'tx_m', 'ty_m', 'tz_m', 'qw', 'qx', 'qy', 'qz']
KEY = ['log_id', 'track_uuid', 'timestamp_ns']

def load_per_log(root):
    parts = []
    for f in tqdm(sorted(Path(root).glob('*/sm_annotations.feather')), desc='logs'):
        df = pd.read_feather(f, columns=COLS)
        df = df[df['category'].isin(VEHICLE_CATS)].copy()
        df['log_id'] = f.parent.name
        parts.append(df)
    return pd.concat(parts, ignore_index=True)

def load_big_feather(path):
    # 설명(prompt)마다 같은 트랙이 반복 저장되어 있으므로 배치 단위로 읽으며 중복 제거
    parts = []
    with pa.memory_map(str(path)) as src:
        reader = ipc.open_file(src)
        cols = [c for c in ['log_id'] + COLS if c in reader.schema.names]
        for i in tqdm(range(reader.num_record_batches), desc='batches'):
            df = pa.Table.from_batches([reader.get_batch(i)]).select(cols).to_pandas()
            df = df[df['category'].isin(VEHICLE_CATS)].drop_duplicates(KEY)
            parts.append(df)
    return pd.concat(parts, ignore_index=True)

ann = load_per_log(SM_LOG_ROOT) if SM_LOG_ROOT is not None else load_big_feather(SM_FEATHER)
ann = ann.drop_duplicates(KEY)
ann['timestamp_ns'] = ann['timestamp_ns'].astype('int64')
ann['category'] = ann['category'].astype('category')
LOG_IDS = sorted(ann['log_id'].unique())
print(f'로그 {len(LOG_IDS)}개, 차량 트랙 {ann.groupby(["log_id", "track_uuid"]).ngroups:,}개, 행 {len(ann):,}')
display(ann['category'].value_counts().to_frame('rows'))

batches:   0%|          | 0/264 [00:00<?, ?it/s]

로그 150개, 차량 트랙 11,819개, 행 1,076,274


,rows
category,
REGULAR_VEHICLE,968752
EGO_VEHICLE,23547
LARGE_VEHICLE,21275
BOX_TRUCK,18566
BUS,14065
TRUCK,9223
MOTORCYCLE,8044
TRUCK_CAB,7414
SCHOOL_BUS,3416


In [14]:
# ==== 3. ego pose (city_SE3_egovehicle.feather) 확보 ====
# 정답 주석 좌표는 ego 기준이므로 속도·방향 계산에는 ego pose로 city 좌표 변환이 필요합니다.
# 로그당 수십 KB이며, 없으면 Argoverse 공개 S3에서 로그인 없이 내려받습니다.
AV2_ROOTS = ['/content/drive/MyDrive/data/datasets/sensor', str(Path.home() / 'data/datasets/sensor')]
POSE_DIR = OUT_DIR / 'ego_poses'
POSE_DIR.mkdir(parents=True, exist_ok=True)

def pose_path(log_id):
    cands = [POSE_DIR / f'{log_id}.feather']
    if SM_LOG_ROOT is not None:
        cands.append(Path(SM_LOG_ROOT) / log_id / 'city_SE3_egovehicle.feather')
    cands += [Path(r) / SPLIT / log_id / 'city_SE3_egovehicle.feather' for r in AV2_ROOTS]
    return next((c for c in cands if c.exists()), None)

missing = [l for l in LOG_IDS if pose_path(l) is None]
print(f'pose 보유 {len(LOG_IDS) - len(missing)}개, 다운로드 필요 {len(missing)}개')
if missing:
    s3 = s3_client()
    def fetch(log_id):
        key = f'datasets/av2/sensor/{SPLIT}/{log_id}/city_SE3_egovehicle.feather'
        try:
            s3.download_file('argoverse', key, str(POSE_DIR / f'{log_id}.feather'))
            return None
        except Exception as e:
            return (log_id, repr(e)[:120])
    with ThreadPoolExecutor(16) as ex:
        fails = [r for r in tqdm(ex.map(fetch, missing), total=len(missing)) if r]
    print('실패:', fails[:5] if fails else '없음')
LOG_IDS = [l for l in LOG_IDS if pose_path(l) is not None]
print('pose 확보 로그 수:', len(LOG_IDS))

pose 보유 150개, 다운로드 필요 0개
pose 확보 로그 수: 150


In [15]:
# ==== 4. 운동 상태 계산 + 사건 구간 검출 ====
def sg_window(n, dt):
    w = int(round(P['SMOOTH_WIN'] / dt)) | 1
    return min(w, n if n % 2 else n - 1)

def runs(mask):
    idx = np.flatnonzero(mask)
    if idx.size == 0:
        return []
    cut = np.flatnonzero(np.diff(idx) > 1)
    return list(zip(np.r_[idx[0], idx[cut + 1]], np.r_[idx[cut], idx[-1]]))

def merge_runs(rs, t, dt, gap):
    out = []
    for a, b in rs:
        if out and t[a] - (t[out[-1][1]] + dt) <= gap + 1e-9:
            out[-1] = (out[-1][0], b)
        else:
            out.append((a, b))
    return out

def to_city(df_log, pose):
    pose = (pose[['timestamp_ns', 'qw', 'qx', 'qy', 'qz', 'tx_m', 'ty_m', 'tz_m']]
            .astype({'timestamp_ns': 'int64'}).sort_values('timestamp_ns')
            .rename(columns=lambda c: c if c == 'timestamp_ns' else 'e_' + c))
    m = pd.merge_asof(df_log.sort_values('timestamp_ns'), pose, on='timestamp_ns',
                      direction='nearest', tolerance=50_000_000).dropna(subset=['e_qw'])
    Re = R.from_quat(m[['e_qx', 'e_qy', 'e_qz', 'e_qw']].to_numpy())
    Ro = R.from_quat(m[['qx', 'qy', 'qz', 'qw']].to_numpy())
    xyz = Re.apply(m[['tx_m', 'ty_m', 'tz_m']].to_numpy()) + m[['e_tx_m', 'e_ty_m', 'e_tz_m']].to_numpy()
    m['x'], m['y'] = xyz[:, 0], xyz[:, 1]
    m['yaw'] = (Re * Ro).as_euler('zyx')[:, 0]
    return m

def kinematics(g, t0):
    # 관측 공백으로 나눈 조각마다 평활화 후 미분
    g = g.sort_values('timestamp_ns')
    t = (g['timestamp_ns'].to_numpy() - t0) / 1e9
    x, y, yaw = g['x'].to_numpy(), g['y'].to_numpy(), np.unwrap(g['yaw'].to_numpy())
    ex, ey = g['tx_m'].to_numpy(), g['ty_m'].to_numpy()
    out = []
    breaks = np.flatnonzero(np.diff(t) > P['SEG_GAP']) + 1
    for sid, idx in enumerate(np.split(np.arange(len(t)), breaks)):
        if len(idx) < P['MIN_SEG_FRAMES']:
            continue
        ts = t[idx]
        dt = float(np.median(np.diff(ts)))
        w = sg_window(len(idx), dt)
        vx = savgol_filter(x[idx], w, 2, deriv=1, delta=dt)
        vy = savgol_filter(y[idx], w, 2, deriv=1, delta=dt)
        sp = np.hypot(vx, vy)
        out.append(pd.DataFrame(dict(
            seg=sid, t=ts, dt=dt,
            x=savgol_filter(x[idx], w, 2), y=savgol_filter(y[idx], w, 2),
            ex=ex[idx], ey=ey[idx], speed=sp,
            a_long=savgol_filter(sp, w, 2, deriv=1, delta=dt),
            yaw=np.degrees(savgol_filter(yaw[idx], w, 2)),
            yawrate=np.degrees(savgol_filter(yaw[idx], w, 2, deriv=1, delta=dt)))))
    return pd.concat(out, ignore_index=True) if out else None

def detect_events(k):
    evs = []
    for sid, s in k.groupby('seg', sort=True):
        t, dt, n = s['t'].to_numpy(), float(s['dt'].iloc[0]), len(s)
        sp, a, yr = s['speed'].to_numpy(), s['a_long'].to_numpy(), s['yawrate'].to_numpy()
        x, y, yw = s['x'].to_numpy(), s['y'].to_numpy(), s['yaw'].to_numpy()

        def add(name, rs, min_dur, **extra):
            for i, j in rs:
                st, en = float(t[i]), float(t[j] + dt)
                if en - st >= min_dur - 1e-9:
                    evs.append(dict(event=name, start=st, end=en, dur=en - st, seg=sid,
                                    cens_start=bool(i == 0), cens_end=bool(j == n - 1), **extra))

        add('STOP', merge_runs(runs(sp < P['V_STOP']), t, dt, P['GAP_FILL']), P['STOP_MIN_DUR'])
        def dv_ok(rs, sign):
            return [(i, j) for i, j in rs if sign * (sp[j] - sp[i]) >= P['MIN_DV']]
        add('DECELERATE', dv_ok(merge_runs(runs((a <= P['A_DECEL']) & (sp >= P['V_STOP'])), t, dt, P['GAP_FILL']), -1), P['ACC_MIN_DUR'])
        add('ACCELERATE', dv_ok(merge_runs(runs(a >= P['A_ACCEL']), t, dt, P['GAP_FILL']), +1), P['ACC_MIN_DUR'])
        for sign, d in [(1, 'left'), (-1, 'right')]:
            for i, j in merge_runs(runs(sign * yr >= P['TURN_YAWRATE']), t, dt, P['TURN_GAP_MERGE']):
                angle = float(yw[j] - yw[i])
                path = float(np.hypot(np.diff(x[i:j + 1]), np.diff(y[i:j + 1])).sum())
                if sign * angle >= P['TURN_ANGLE'] and path >= P['TURN_MIN_DIST']:
                    add('TURN', [(i, j)], 0.0, direction=d, angle=round(angle, 1))
    return evs

KIN, EGO, T0 = {}, {}, {}
event_rows, n_parked, n_tracks = [], 0, 0
for log_id in tqdm(LOG_IDS, desc='logs'):
    pose = pd.read_feather(pose_path(log_id))
    df_log = ann[ann['log_id'] == log_id].drop(columns='log_id')
    t0 = int(df_log['timestamp_ns'].min()); T0[log_id] = t0
    ts_ann = np.sort(df_log['timestamp_ns'].unique())
    df_log = df_log.astype({'category': str})
    is_ego = df_log['category'] == 'EGO_VEHICLE'
    if not INCLUDE_EGO:
        df_log = df_log[~is_ego]
    elif not is_ego.any():
        ego = pd.DataFrame(dict(track_uuid='ego', category='EGO_VEHICLE', timestamp_ns=ts_ann,
                                tx_m=0.0, ty_m=0.0, tz_m=0.0, qw=1.0, qx=0.0, qy=0.0, qz=0.0))
        df_log = pd.concat([df_log, ego], ignore_index=True)
    m = to_city(df_log, pose)
    e = m[m['category'] == 'EGO_VEHICLE']
    EGO[log_id] = e[['timestamp_ns', 'x', 'y']].assign(t=(e['timestamp_ns'] - t0) / 1e9) if len(e) else None
    for uuid, g in m.groupby('track_uuid', sort=False):
        n_tracks += 1
        if np.linalg.norm(g[['x', 'y']].max().to_numpy() - g[['x', 'y']].min().to_numpy()) < P['PARKED_DISP']:
            n_parked += 1
            continue
        k = kinematics(g, t0)
        if k is None:
            continue
        KIN[(log_id, uuid)] = k
        cat = str(g['category'].iloc[0])
        for ev in detect_events(k):
            event_rows.append(dict(log_id=log_id, track_uuid=uuid, category=cat, **ev))

events = pd.DataFrame(event_rows)
for c in ['direction', 'angle']:
    if c not in events:
        events[c] = np.nan
print(f'트랙 {n_tracks:,}개 중 주차 {n_parked:,}개 제외, 운동 트랙 {len(KIN):,}개')
by = events.groupby('event')
summ = pd.DataFrame({
    'intervals': by.size(),
    'tracks': events.drop_duplicates(['log_id', 'track_uuid', 'event']).groupby('event').size(),
    'logs': by['log_id'].nunique(),
    'dur_median_s': by['dur'].median(),
    'censored_ratio': (events['cens_start'] | events['cens_end']).groupby(events['event']).mean(),
})
print('censored_ratio = 시작 또는 끝이 관측 경계에서 잘린 구간의 비율')
display(summ.round(2))

logs:   0%|          | 0/150 [00:00<?, ?it/s]

트랙 11,819개 중 주차 8,027개 제외, 운동 트랙 3,737개
censored_ratio = 시작 또는 끝이 관측 경계에서 잘린 구간의 비율


,intervals,tracks,logs,dur_median_s,censored_ratio
event,,,,,
ACCELERATE,1073,945,140,2.1,0.06
DECELERATE,1114,979,147,2.1,0.09
STOP,1016,885,134,3.8,0.69
TURN,479,475,127,4.7,0.52


In [16]:
# ==== 5. 같은 트랙 안의 사건 쌍 관계 계산 ====
# 구간 I=[s,e). first/second = 먼저/나중에 시작한 사건, gap = second.start - first.end
#   OVERLAP: 교집합 > MIN_OVERLAP
#   MEETS  : -MIN_OVERLAP <= gap <= MEET_TOL (맞닿음)
#   BEFORE : gap > MEET_TOL
# boundary_cut: 관계 판정에 필요한 경계(first의 끝, second의 시작)가 관측 경계에서 잘린 경우.
#   예) 로그 끝까지 정지해 있으면 정지의 '끝'은 잘렸지만 '시작'은 관측됨 -> 회전 후 정지 판정에는 문제없음
#   OVERLAP은 교집합 자체를 관측했으므로 잘림 여부와 무관
inst_rows = []
inst['is_ego'] = inst['category'] == 'EGO_VEHICLE'
for (log_id, uuid), g in events.groupby(['log_id', 'track_uuid'], sort=False):
    E = g.sort_values(['start', 'event']).to_dict('records')
    for i in range(len(E)):
        for j in range(i + 1, len(E)):
            p, q = E[i], E[j]
            if p['event'] == q['event']:
                continue
            ov = min(p['end'], q['end']) - max(p['start'], q['start'])
            gap = q['start'] - p['end']
            if ov > P['MIN_OVERLAP'] + 1e-9:
                rel, cut = 'OVERLAP', False
            else:
                rel = 'MEETS' if gap <= P['MEET_TOL'] else 'BEFORE'
                cut = bool(p['cens_end'] or q['cens_start'])
            inst_rows.append(dict(
                log_id=log_id, track_uuid=uuid, category=p['category'],
                first=p['event'], first_start=p['start'], first_end=p['end'], first_dir=p['direction'],
                second=q['event'], second_start=q['start'], second_end=q['end'], second_dir=q['direction'],
                relation=rel, gap=round(gap, 2), overlap=round(max(ov, 0.0), 2),
                boundary_cut=cut))
inst = pd.DataFrame(inst_rows)
inst['track_key'] = inst['log_id'] + '/' + inst['track_uuid']
print('사건 쌍 인스턴스:', len(inst))
display(inst['relation'].value_counts().to_frame('instances'))

사건 쌍 인스턴스: 1693


,instances
relation,
BEFORE,1100
MEETS,327
OVERLAP,266


In [17]:
# ==== 6. 요약표: 관계별 (트랙 수 / 서로 다른 로그 수) ====
def fmt(sub):
    return f"{sub['track_key'].nunique()} / {sub['log_id'].nunique()}"

def summarize_compact(inst, delta, exclude_cut=False):
    df = inst[~inst['boundary_cut']] if exclude_cut else inst
    rows = []
    for X, Y in combinations(EVENT_TYPES, 2):
        sub = df[((df['first'] == X) & (df['second'] == Y)) | ((df['first'] == Y) & (df['second'] == X))]
        def cell(mask): return fmt(sub[mask])
        rows.append({'X': X, 'Y': Y, 'both': fmt(sub),
                     'X→Y BEFORE': cell((sub['first'] == X) & (sub['relation'] == 'BEFORE') & (sub['gap'] <= delta)),
                     'Y→X BEFORE': cell((sub['first'] == Y) & (sub['relation'] == 'BEFORE') & (sub['gap'] <= delta)),
                     'X→Y MEETS': cell((sub['first'] == X) & (sub['relation'] == 'MEETS')),
                     'Y→X MEETS': cell((sub['first'] == Y) & (sub['relation'] == 'MEETS')),
                     'OVERLAP': cell(sub['relation'] == 'OVERLAP')})
    return pd.DataFrame(rows)

full = summarize_compact(inst, P['DELTA_MAIN'])
clean = summarize_compact(inst, P['DELTA_MAIN'], exclude_cut=True)

def feasibility(inst, delta, exclude_cut):
    df = inst[~inst['boundary_cut']] if exclude_cut else inst
    rows = []
    for X, Y in combinations(EVENT_TYPES, 2):
        b = df[df['relation'] == 'BEFORE']
        nxy = b[(b['first'] == X) & (b['second'] == Y) & (b['gap'] <= delta)]['log_id'].nunique()
        nyx = b[(b['first'] == Y) & (b['second'] == X) & (b['gap'] <= delta)]['log_id'].nunique()
        o = df[(df['relation'] == 'OVERLAP') & (((df['first'] == X) & (df['second'] == Y)) | ((df['first'] == Y) & (df['second'] == X)))]
        no = o['log_id'].nunique()
        rows.append(dict(X=X, Y=Y, logs_XbeforeY=nxy, logs_YbeforeX=nyx, logs_overlap=no,
                         ok=min(nxy, nyx, no) >= P['MIN_LOGS_FEASIBLE']))
    return pd.DataFrame(rows)

print(f"표기: 트랙 수 / 로그 수, BEFORE는 간격 <= {P['DELTA_MAIN']} s")
print('\n[전체 구간]'); display(full)
print('\n[판정 경계가 잘리지 않은 인스턴스만]'); display(clean)
print(f"\n[대조쌍 구성 가능성: BEFORE 양방향과 OVERLAP 모두 서로 다른 로그 {P['MIN_LOGS_FEASIBLE']}개 이상]")
display(feasibility(inst, P['DELTA_MAIN'], exclude_cut=True))

# Δ 민감도 (로그 수 기준)
sens = []
for d in P['DELTAS']:
    f = feasibility(inst, d, exclude_cut=True)
    f['delta'] = d
    sens.append(f[['X', 'Y', 'delta', 'logs_XbeforeY', 'logs_YbeforeX']])
print('\n[BEFORE 최대 간격 Δ 민감도, 경계가 잘리지 않은 인스턴스]')
display(pd.concat(sens).pivot_table(index=['X', 'Y'], columns='delta', values=['logs_XbeforeY', 'logs_YbeforeX']))

print('\n[같은 조건, ego 제외]')
display(feasibility(inst[~inst['is_ego']], P['DELTA_MAIN'], exclude_cut=True))

def exclusive(inst, delta):
    rows = []
    for X, Y in combinations(EVENT_TYPES, 2):
        sub = inst[((inst['first'] == X) & (inst['second'] == Y)) | ((inst['first'] == Y) & (inst['second'] == X))]
        near = sub[(sub['relation'] != 'BEFORE') | (sub['gap'] <= delta)].copy()
        near['lab'] = np.where(near['relation'] == 'OVERLAP', 'OVERLAP',
                       np.where(near['relation'] == 'MEETS', 'MEETS',
                       np.where(near['first'] == X, 'X→Y', 'Y→X')))
        labs = near.groupby('track_key')['lab'].agg(lambda s: frozenset(s))
        only = labs[labs.map(len) == 1].map(lambda s: next(iter(s)))
        row = {'X': X, 'Y': Y, 'mixed_tracks': int((labs.map(len) > 1).sum())}
        for lab in ['X→Y', 'Y→X', 'OVERLAP', 'MEETS']:
            keys = only[only == lab].index
            row[f'only {lab}'] = f"{len(keys)} / {len({k.split('/')[0] for k in keys})}"
        rows.append(row)
    return pd.DataFrame(rows)

print(f"\n[관계가 하나뿐인 트랙 (트랙 / 로그), Δ={P['DELTA_MAIN']} s]")
display(exclusive(inst, P['DELTA_MAIN']))

# BEFORE 간격 분포
b = inst[(inst['relation'] == 'BEFORE') & ~inst['boundary_cut']]
print('\n[BEFORE 간격(s) 분포, 방향별]')
display(b.groupby(['first', 'second'])['gap'].describe(percentiles=[.25, .5, .75]).round(2))

표기: 트랙 수 / 로그 수, BEFORE는 간격 <= 3.0 s

[전체 구간]


,X,Y,both,X→Y BEFORE,Y→X BEFORE,X→Y MEETS,Y→X MEETS,OVERLAP
0,TURN,STOP,100 / 57,5 / 5,49 / 34,1 / 1,8 / 8,5 / 5
1,TURN,DECELERATE,156 / 86,18 / 16,29 / 23,2 / 2,10 / 10,66 / 51
2,TURN,ACCELERATE,220 / 92,11 / 9,19 / 14,3 / 3,5 / 5,179 / 87
3,STOP,DECELERATE,274 / 108,5 / 5,120 / 63,0 / 0,115 / 66,0 / 0
4,STOP,ACCELERATE,294 / 95,132 / 60,4 / 4,122 / 65,0 / 0,0 / 0
5,DECELERATE,ACCELERATE,301 / 112,89 / 61,124 / 59,32 / 6,15 / 7,0 / 0



[판정 경계가 잘리지 않은 인스턴스만]


,X,Y,both,X→Y BEFORE,Y→X BEFORE,X→Y MEETS,Y→X MEETS,OVERLAP
0,TURN,STOP,100 / 57,5 / 5,49 / 34,1 / 1,8 / 8,5 / 5
1,TURN,DECELERATE,156 / 86,18 / 16,29 / 23,2 / 2,10 / 10,66 / 51
2,TURN,ACCELERATE,220 / 92,11 / 9,19 / 14,3 / 3,5 / 5,179 / 87
3,STOP,DECELERATE,274 / 108,5 / 5,120 / 63,0 / 0,115 / 66,0 / 0
4,STOP,ACCELERATE,294 / 95,132 / 60,4 / 4,122 / 65,0 / 0,0 / 0
5,DECELERATE,ACCELERATE,301 / 112,89 / 61,124 / 59,32 / 6,15 / 7,0 / 0



[대조쌍 구성 가능성: BEFORE 양방향과 OVERLAP 모두 서로 다른 로그 5개 이상]


,X,Y,logs_XbeforeY,logs_YbeforeX,logs_overlap,ok
0,TURN,STOP,5,34,5,True
1,TURN,DECELERATE,16,23,51,True
2,TURN,ACCELERATE,9,14,87,True
3,STOP,DECELERATE,5,63,0,False
4,STOP,ACCELERATE,60,4,0,False
5,DECELERATE,ACCELERATE,61,59,0,False



[BEFORE 최대 간격 Δ 민감도, 경계가 잘리지 않은 인스턴스]


logs_XbeforeY             logs_YbeforeX            
delta                           1.0   3.0   5.0           1.0   3.0   5.0
X          Y                                                             
DECELERATE ACCELERATE          24.0  61.0  67.0          32.0  59.0  81.0
STOP       ACCELERATE          40.0  60.0  68.0           0.0   4.0  10.0
           DECELERATE           0.0   5.0   7.0          42.0  63.0  75.0
TURN       ACCELERATE           5.0   9.0  12.0           4.0  14.0  20.0
           DECELERATE           3.0  16.0  30.0           7.0  23.0  26.0
           STOP                 0.0   5.0   7.0          18.0  34.0  41.0


[같은 조건, ego 제외]


KeyError: 'is_ego'

In [ ]:
# ==== 7. 결과 저장 ====
events.to_csv(OUT_DIR / 'events.csv', index=False)
inst.to_csv(OUT_DIR / 'instances.csv', index=False)
full.to_csv(OUT_DIR / f'summary_all_delta{P["DELTA_MAIN"]}.csv', index=False)
clean.to_csv(OUT_DIR / f'summary_exclude_cut_delta{P["DELTA_MAIN"]}.csv', index=False)
feasibility(inst, P['DELTA_MAIN'], True).to_csv(OUT_DIR / 'feasibility.csv', index=False)
pd.concat([k.assign(log_id=l, track_uuid=u) for (l, u), k in KIN.items()],
          ignore_index=True).to_feather(OUT_DIR / 'kinematics.feather')
json.dump(dict(P=P, EVENT_TYPES=EVENT_TYPES, SPLIT=SPLIT, INCLUDE_EGO=INCLUDE_EGO,
               n_logs=len(LOG_IDS), source=str(SM_LOG_ROOT or SM_FEATHER)),
          open(OUT_DIR / 'params.json', 'w'), indent=2, ensure_ascii=False)
print('저장 완료:', sorted(p.name for p in OUT_DIR.iterdir()))

## 데이터 보는 방법 1: 궤적과 운동 상태 그래프

카메라 없이도 GT를 검수할 수 있는 기본 방법입니다. 왼쪽은 위에서 내려다본 city 좌표 궤적(BEV), 오른쪽은 속력·종방향 가속도·요레이트의 시간 변화이며, 두 사건 구간을 색으로 칠합니다. 사건 정의가 사람의 판단과 맞는지 여기서 먼저 확인하십시오.

In [ ]:
# ==== 8. 인스턴스 시각화 ====
import matplotlib.pyplot as plt

def pick(first=None, second=None, relation=None, exclude_cut=True, n=5, seed=0):
    df = inst[~inst['boundary_cut']] if exclude_cut else inst
    if first: df = df[df['first'] == first]
    if second: df = df[df['second'] == second]
    if relation: df = df[df['relation'] == relation]
    return df.sample(min(n, len(df)), random_state=seed) if len(df) else df

def show_instance(row, pad=3.0):
    k = KIN[(row['log_id'], row['track_uuid'])]
    lo, hi = min(row['first_start'], row['second_start']) - pad, max(row['first_end'], row['second_end']) + pad
    w = k[(k['t'] >= lo) & (k['t'] <= hi)]
    fig = plt.figure(figsize=(14, 6))
    gs = fig.add_gridspec(3, 2, width_ratios=[1, 1.4])
    ax = fig.add_subplot(gs[:, 0])
    ego = EGO.get(row['log_id'])
    if ego is not None and row['track_uuid'] != 'ego':
        ew = ego[(ego['t'] >= lo) & (ego['t'] <= hi)]
        ax.plot(ew['x'], ew['y'], color='0.5', lw=2, alpha=.5, label='ego path (window)')
        if len(ew):
            ax.plot(ew['x'].iloc[0], ew['y'].iloc[0], 's', color='0.5', ms=6)
    ax.plot(k['x'], k['y'], color='0.6', lw=1, ls=':', label='track (whole)')
    ax.scatter(w['x'], w['y'], c=w['t'], cmap='viridis', s=8, label='track (window, color=time)')
    for name, s, e, c in [(row['first'], row['first_start'], row['first_end'], 'tab:blue'),
                          (row['second'], row['second_start'], row['second_end'], 'tab:orange')]:
        seg = k[(k['t'] >= s) & (k['t'] < e)]
        ax.plot(seg['x'], seg['y'], color=c, lw=4, alpha=.7, label=f'{name} [{s:.1f}, {e:.1f})')
    ax.plot(w['x'].iloc[0], w['y'].iloc[0], 'k^', ms=8, label='window start')
    cx, cy = (w['x'].min() + w['x'].max()) / 2, (w['y'].min() + w['y'].max()) / 2
    half = max(w['x'].max() - w['x'].min(), w['y'].max() - w['y'].min()) / 2 + 15
    ax.set_xlim(cx - half, cx + half); ax.set_ylim(cy - half, cy + half)
    ax.set_aspect('equal'); ax.set_title('BEV (city frame, m)'); ax.legend(fontsize=7, loc='best')
    for r, (col, ylab, ths) in enumerate([('speed', 'speed [m/s]', [P['V_STOP']]),
                                          ('a_long', 'a_long [m/s²]', [P['A_DECEL'], P['A_ACCEL']]),
                                          ('yawrate', 'yaw rate [deg/s]', [-P['TURN_YAWRATE'], P['TURN_YAWRATE']])]):
        a2 = fig.add_subplot(gs[r, 1])
        for _, sseg in w.groupby('seg'):
            a2.plot(sseg['t'], sseg[col], color='k', lw=1.2)
        for th in ths:
            a2.axhline(th, color='r', ls='--', lw=.8)
        a2.axvspan(row['first_start'], row['first_end'], color='tab:blue', alpha=.2)
        a2.axvspan(row['second_start'], row['second_end'], color='tab:orange', alpha=.2)
        a2.set_ylabel(ylab); a2.grid(alpha=.3)
        if r == 2: a2.set_xlabel('time since log start [s]')
    fig.suptitle(f"{row['first']} → {row['second']}  {row['relation']}  gap={row['gap']} s  overlap={row['overlap']} s\n"
                 f"log {row['log_id'][:8]}  track {str(row['track_uuid'])[:8]}  ({row['category']})", fontsize=10)
    plt.tight_layout(); plt.show()

# 예시: 회전 후 정지(BEFORE), 정지 후 회전(BEFORE), 회전과 감속의 OVERLAP
for f, s, rel in [('TURN', 'STOP', 'BEFORE'), ('STOP', 'TURN', 'BEFORE'), (None, None, 'OVERLAP')]:
    ex = pick(f, s, rel, n=1)
    if len(ex):
        show_instance(ex.iloc[0])
    else:
        print('해당 인스턴스 없음:', f, s, rel)

## 데이터 보는 방법 2 (선택): 해당 시점의 카메라 이미지

GT 검수용입니다. **모델 입력에는 쓰지 않습니다.** 대상 차량의 ego 기준 방향으로 가장 가까운 링 카메라를 고르고, 사건 구간 경계 시점의 이미지만 공개 S3에서 내려받습니다(장당 수백 KB). 이미지에 박스를 그리지는 않으므로 제목의 ego 기준 위치(앞 x m, 왼쪽 y m)로 대상을 찾으십시오.

In [ ]:
# ==== 9. (선택) 카메라 이미지로 검수 ====
import io
from PIL import Image

# 링 카메라의 대략적인 방위각(도, ego 기준 반시계 +). 카메라 선택용 근사값입니다.
CAMS = {'ring_front_center': 0, 'ring_front_left': 45, 'ring_front_right': -45,
        'ring_side_left': 100, 'ring_side_right': -100, 'ring_rear_left': 150, 'ring_rear_right': -150}
_cam_index = {}

def cam_timestamps(log_id, cam, s3):
    key = (log_id, cam)
    if key not in _cam_index:
        prefix = f'datasets/av2/sensor/{SPLIT}/{log_id}/sensors/cameras/{cam}/'
        ts = []
        for page in s3.get_paginator('list_objects_v2').paginate(Bucket='argoverse', Prefix=prefix):
            ts += [int(o['Key'].rsplit('/', 1)[1].split('.')[0]) for o in page.get('Contents', [])]
        _cam_index[key] = np.array(sorted(ts), dtype=np.int64)
    return _cam_index[key]

def show_camera(row, n_extra=0):
    s3 = s3_client()
    k = KIN[(row['log_id'], row['track_uuid'])]
    times = [row['first_start'], row['first_end'], row['second_start'], row['second_end']]
    fig, axes = plt.subplots(1, len(times), figsize=(5 * len(times), 4.2))
    for ax, t in zip(axes, times):
        r = k.iloc[(k['t'] - t).abs().argmin()]
        if row['track_uuid'] == 'ego':
            cam, info = 'ring_front_center', 'ego vehicle'
        else:
            bearing = np.degrees(np.arctan2(r['ey'], r['ex']))
            cam = min(CAMS, key=lambda c: abs((bearing - CAMS[c] + 180) % 360 - 180))
            info = f"ahead {r['ex']:.1f} m, left {r['ey']:.1f} m"
        ts = cam_timestamps(row['log_id'], cam, s3)
        t_ns = T0[row['log_id']] + int(t * 1e9)
        nearest = int(ts[np.abs(ts - t_ns).argmin()])
        obj = s3.get_object(Bucket='argoverse',
                            Key=f'datasets/av2/sensor/{SPLIT}/{row["log_id"]}/sensors/cameras/{cam}/{nearest}.jpg')
        ax.imshow(Image.open(io.BytesIO(obj['Body'].read())))
        ax.set_title(f't={t:.1f}s  {cam}\n{info}  speed {r["speed"]:.1f} m/s', fontsize=8)
        ax.axis('off')
    fig.suptitle(f"{row['first']} → {row['second']}  {row['relation']}  (boundaries: first start/end, second start/end)", fontsize=10)
    plt.tight_layout(); plt.show()

ex = pick('TURN', 'STOP', 'BEFORE', n=1)
if len(ex):
    show_camera(ex.iloc[0])